# PyTorch Tutorial: Music Generation Models

Music generation sits at the intersection of sequence modeling, signal processing, and creative control. This notebook is **theory-first** with optional lightweight code that runs on CPU, so you can understand modern research direction while still building intuition with runnable from-scratch models.

We cover two practical representations:
- **Symbolic tokens (MIDI-like)** for structure, composition, and lower compute.
- **Raw audio tokens** for timbre and end-to-end waveform generation.


## Learning Objectives

By the end of this notebook, you should be able to:

1. Explain why representation choice (symbolic vs audio tokens) changes model design.
2. Describe the research trajectory from Music Transformer to modern text-to-music systems.
3. Train a tiny **Transformer** for symbolic music token generation.
4. Train a tiny **autoregressive waveform-token model** with mu-law quantization.
5. Evaluate generated outputs with lightweight quantitative proxies and listening heuristics.
6. Reason about control, failure modes, and licensing/data constraints in production settings.


## 1. Problem Framing: What Is a Music Generation Model?

A music generation model learns to predict musical structure over time. Depending on representation, it may model:

- **Composition-level structure**: motifs, harmony, phrase repetition.
- **Performance detail**: velocity, timing micro-variation, articulation.
- **Timbre/acoustics**: instrument color and texture in waveform space.

Core objective (autoregressive form):

\[
P(x_{1:T}) = \prod_{t=1}^{T} P(x_t \mid x_{<t})
\]

Modern systems often add conditioning signals (text, genre, tempo, chords, stems) and use tokenizers/compression models to make long audio sequences tractable.


## 2. Representation Choices

| Representation | Typical Token | Pros | Tradeoffs |
|---|---|---|---|
| Symbolic (MIDI-like) | Note events, durations, control tokens | Efficient, easy to train, clearer musical structure | No direct timbre realism |
| Spectrogram | Time-frequency bins | Good for perceptual features | Phase/vocoder complexity |
| Raw audio (compressed tokens) | Codec/tokenizer indices | End-to-end timbre + structure | Long sequences, higher compute |

A pragmatic learning path is symbolic first (fast feedback), then audio tokens for realism.


## 3. Research Landscape (Timeline)

| Model | Year | Core Idea | Representation |
|---|---:|---|---|
| MusicVAE | 2018 | Latent interpolation for musical phrases | Symbolic |
| Music Transformer | 2018 | Relative attention for long-range structure | Symbolic |
| Jukebox | 2020 | VQ-VAE + autoregressive hierarchy for raw audio | Audio |
| MusicLM | 2023 | Text-to-music with hierarchical semantic/acoustic modeling | Audio |
| MusicGen | 2023 | Single-stage autoregressive text-to-music over EnCodec-like tokens | Audio |
| MAGNeT | 2024 | Masked non-autoregressive decoding for faster generation | Audio |
| Stable Audio Open | 2024 | Open text-to-audio/music generation model | Audio |
| MusicRL | 2024 | Preference optimization for better musical quality/alignment | Audio |
| Live / Real-Time Music Models | 2025 | Interactive low-latency generation/control | Audio + control streams |

This notebook does **from-scratch toy modeling** only (no pretrained downloads).


## 4. Symbolic Generation from Scratch (Tiny Transformer)

We start with synthetic symbolic sequences that mimic motifs and transpositions. This keeps training lightweight while still teaching sequence modeling fundamentals.


In [ ]:
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt


def set_seed(seed: int = 42) -> None:
    """Set random seeds for reproducible CPU demos."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
def generate_symbolic_dataset(
    num_samples: int,
    seq_len: int,
    vocab_size: int,
) -> torch.Tensor:
    """Create simple motif-based token sequences with transposition and rhythm variation."""
    rng = np.random.default_rng(123)

    motifs = [
        np.array([0, 2, 4, 7, 4, 2], dtype=np.int64),
        np.array([0, 3, 5, 7, 5, 3], dtype=np.int64),
        np.array([0, 2, 5, 9, 5, 2], dtype=np.int64),
    ]

    data = np.zeros((num_samples, seq_len), dtype=np.int64)
    for i in range(num_samples):
        motif = motifs[rng.integers(0, len(motifs))]
        transpose = int(rng.integers(-4, 5))
        step_jitter = int(rng.integers(-1, 2))

        seq = []
        while len(seq) < seq_len:
            for note in motif:
                shifted = note + transpose + step_jitter
                token = int(np.clip(shifted + vocab_size // 3, 0, vocab_size - 1))
                seq.append(token)
                if rng.random() < 0.18:
                    seq.append(token)
                if len(seq) >= seq_len:
                    break

        data[i] = np.array(seq[:seq_len], dtype=np.int64)

    return torch.tensor(data, dtype=torch.long)


def synthesize_tone_sequence(token_seq: np.ndarray, sample_rate: int = 16000) -> np.ndarray:
    """Render symbolic tokens into a simple monophonic waveform for quick listening/plotting."""
    token_duration = 0.055
    attack = 0.15
    decay = 0.2

    wave = []
    for tok in token_seq:
        n = int(token_duration * sample_rate)
        t = np.linspace(0.0, token_duration, n, endpoint=False)

        semitone = (int(tok) % 36) - 12
        f0 = 220.0 * (2.0 ** (semitone / 12.0))

        tone = (
            np.sin(2 * np.pi * f0 * t)
            + 0.35 * np.sin(2 * np.pi * 2 * f0 * t)
            + 0.15 * np.sin(2 * np.pi * 3 * f0 * t)
        )

        env = np.ones_like(t)
        a = max(1, int(attack * len(t)))
        d = max(1, int(decay * len(t)))
        env[:a] = np.linspace(0.0, 1.0, a)
        env[-d:] = np.linspace(1.0, 0.2, d)

        wave.append((0.22 * tone * env).astype(np.float32))

    out = np.concatenate(wave) if wave else np.zeros(1, dtype=np.float32)
    out = np.clip(out, -1.0, 1.0)
    return out


vocab_size = 48
seq_len = 64
symbolic_data = generate_symbolic_dataset(num_samples=384, seq_len=seq_len, vocab_size=vocab_size)
print("Symbolic dataset:", symbolic_data.shape)

plt.figure(figsize=(10, 2.5))
plt.plot(symbolic_data[0].numpy(), lw=1.5)
plt.title("Example symbolic token sequence")
plt.xlabel("Step")
plt.ylabel("Token")
plt.grid(alpha=0.2)
plt.show()


In [ ]:
class TinyMusicTransformer(nn.Module):
    """Minimal causal Transformer for token-level symbolic generation."""

    def __init__(
        self,
        vocab_size: int,
        d_model: int = 96,
        nhead: int = 4,
        num_layers: int = 3,
        dim_feedforward: int = 192,
        dropout: float = 0.1,
        max_seq_len: int = 256,
    ):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Parameter(torch.zeros(1, max_seq_len, d_model))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, t = x.shape
        if t > self.max_seq_len:
            x = x[:, -self.max_seq_len :]
            t = x.shape[1]

        h = self.token_emb(x) + self.pos_emb[:, :t, :]
        causal_mask = torch.triu(torch.ones(t, t, device=x.device, dtype=torch.bool), diagonal=1)
        h = self.transformer(h, mask=causal_mask)
        return self.head(self.norm(h))


In [ ]:
def train_symbolic(
    model: nn.Module,
    data: torch.Tensor,
    steps: int = 80,
    lr: float = 3e-3,
    batch_size: int = 32,
) -> list[float]:
    """Train symbolic next-token prediction with short CPU-friendly loops."""
    model.train()
    data = data.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    losses: list[float] = []

    n = data.size(0)
    for _ in range(steps):
        idx = torch.randint(0, n, (batch_size,), device=device)
        batch = data[idx]
        inp = batch[:, :-1]
        tgt = batch[:, 1:]

        logits = model(inp)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        losses.append(float(loss.item()))

    return losses


def sample_symbolic(
    model: nn.Module,
    prompt: torch.Tensor,
    max_new_tokens: int,
    temperature: float = 1.0,
    top_k: int | None = 8,
) -> torch.Tensor:
    """Autoregressive symbolic sampling with optional top-k truncation."""
    model.eval()
    generated = prompt.clone().to(device)

    with torch.inference_mode():
        for _ in range(max_new_tokens):
            ctx = generated[-model.max_seq_len :]
            logits = model(ctx.unsqueeze(0))[0, -1]
            logits = logits / max(temperature, 1e-6)

            if top_k is not None and top_k < logits.numel():
                vals, ids = torch.topk(logits, top_k)
                probs = torch.softmax(vals, dim=-1)
                next_id = ids[torch.multinomial(probs, 1)]
            else:
                probs = torch.softmax(logits, dim=-1)
                next_id = torch.multinomial(probs, 1)

            generated = torch.cat([generated, next_id.view(1)], dim=0)

    return generated.cpu()


symbolic_model = TinyMusicTransformer(vocab_size=vocab_size).to(device)
losses = train_symbolic(symbolic_model, symbolic_data, steps=90, lr=3e-3, batch_size=32)

prompt = symbolic_data[0, :12].cpu()
generated = sample_symbolic(symbolic_model, prompt, max_new_tokens=36, temperature=0.9, top_k=8)

assert generated.numel() == prompt.numel() + 36
print("Generated symbolic length check passed.")

plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.title("TinyMusicTransformer training loss")
plt.xlabel("Step")
plt.ylabel("Cross-entropy")
plt.grid(alpha=0.25)
plt.show()


In [ ]:
set_seed(7)
s1 = sample_symbolic(symbolic_model, prompt, max_new_tokens=24, temperature=0.85, top_k=6)
set_seed(7)
s2 = sample_symbolic(symbolic_model, prompt, max_new_tokens=24, temperature=0.85, top_k=6)
assert torch.equal(s1, s2), "Sampling should be deterministic when seed and model are fixed."
print("Determinism check passed.")

symbolic_audio = synthesize_tone_sequence(generated.numpy(), sample_rate=16000)

fig, axes = plt.subplots(2, 1, figsize=(10, 4.5), constrained_layout=True)
axes[0].plot(generated.numpy(), lw=1.2)
axes[0].set_title("Generated symbolic token sequence")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Token")
axes[0].grid(alpha=0.2)

axes[1].plot(symbolic_audio[:5000], lw=1.0)
axes[1].set_title("Synthesized waveform preview (from symbolic output)")
axes[1].set_xlabel("Sample")
axes[1].set_ylabel("Amplitude")
axes[1].grid(alpha=0.2)
plt.show()


## 5. Raw-Audio Generation from Scratch (Tiny Wave AR)

Now we switch to waveform-token modeling with **mu-law quantization**. This is still tiny and educational, but it demonstrates the core recipe behind autoregressive audio token models.


In [ ]:
def mu_law_encode(wave: torch.Tensor, quantization_channels: int = 256) -> torch.Tensor:
    """Mu-law companding + quantization to discrete audio tokens."""
    mu = quantization_channels - 1
    wave = torch.clamp(wave, -1.0, 1.0)
    magnitude = torch.log1p(mu * torch.abs(wave)) / math.log1p(mu)
    signal = torch.sign(wave) * magnitude
    encoded = ((signal + 1.0) * 0.5 * mu + 0.5).long()
    return torch.clamp(encoded, 0, mu)


def mu_law_decode(tokens: torch.Tensor, quantization_channels: int = 256) -> torch.Tensor:
    """Inverse mu-law decode from token indices to waveform range [-1, 1]."""
    mu = quantization_channels - 1
    signal = 2.0 * (tokens.float() / mu) - 1.0
    magnitude = (torch.pow(1.0 + mu, torch.abs(signal)) - 1.0) / mu
    wave = torch.sign(signal) * magnitude
    return torch.clamp(wave, -1.0, 1.0)


def build_wave_training_tokens(symbolic_sequences: torch.Tensor, sample_rate: int = 8000) -> torch.Tensor:
    """Render several symbolic clips and convert to a single token stream."""
    chunks = []
    for seq in symbolic_sequences[:24]:
        wav = synthesize_tone_sequence(seq.numpy(), sample_rate=sample_rate)
        chunks.append(torch.tensor(wav, dtype=torch.float32))
    waveform = torch.cat(chunks, dim=0)
    tokens = mu_law_encode(waveform, quantization_channels=256)
    return tokens


sample_rate = 8000
wave_tokens = build_wave_training_tokens(symbolic_data, sample_rate=sample_rate)
print("Wave token stream length:", int(wave_tokens.numel()))


In [ ]:
class TinyWaveAR(nn.Module):
    """Tiny autoregressive model over mu-law audio tokens."""

    def __init__(
        self,
        quantization_channels: int = 256,
        emb_dim: int = 64,
        hidden_dim: int = 96,
        context_len: int = 128,
    ):
        super().__init__()
        self.context_len = context_len
        self.quantization_channels = quantization_channels

        self.embedding = nn.Embedding(quantization_channels, emb_dim)
        self.causal_conv = nn.Conv1d(emb_dim, emb_dim, kernel_size=5, padding=4)
        self.gru = nn.GRU(input_size=emb_dim, hidden_size=hidden_dim, batch_first=True)
        self.head = nn.Linear(hidden_dim, quantization_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(x).transpose(1, 2)
        conv = self.causal_conv(emb)[:, :, : x.size(1)]
        h = F.gelu(conv).transpose(1, 2)
        h, _ = self.gru(h)
        return self.head(h)


In [ ]:
def train_wave(
    model: nn.Module,
    token_data: torch.Tensor,
    steps: int = 70,
    lr: float = 2e-3,
    batch_size: int = 24,
) -> list[float]:
    """Train next-token prediction on one long token stream."""
    model.train()
    tokens = token_data.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    losses: list[float] = []

    max_start = tokens.numel() - (model.context_len + 1)
    if max_start <= 1:
        raise ValueError("Token stream too short for the chosen context length.")

    for _ in range(steps):
        starts = torch.randint(0, max_start, (batch_size,), device=device)
        seq_batch = torch.stack([tokens[s : s + model.context_len + 1] for s in starts], dim=0)

        inp = seq_batch[:, :-1]
        tgt = seq_batch[:, 1:]

        logits = model(inp)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        losses.append(float(loss.item()))

    return losses


def sample_wave(
    model: nn.Module,
    prompt_tokens: torch.Tensor,
    num_new_tokens: int,
    temperature: float = 1.0,
) -> torch.Tensor:
    """Autoregressive waveform-token sampling."""
    model.eval()
    generated = prompt_tokens.clone().to(device)

    with torch.inference_mode():
        for _ in range(num_new_tokens):
            ctx = generated[-model.context_len :]
            logits = model(ctx.unsqueeze(0))[0, -1] / max(temperature, 1e-6)
            probs = torch.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, 1)
            generated = torch.cat([generated, next_tok.view(1)], dim=0)

    return generated.cpu()


wave_model = TinyWaveAR(quantization_channels=256, context_len=128).to(device)
wave_losses = train_wave(wave_model, wave_tokens, steps=75, lr=2e-3, batch_size=24)

wave_prompt = wave_tokens[:64]
wave_generated_tokens = sample_wave(wave_model, wave_prompt, num_new_tokens=512, temperature=0.95)
wave_generated = mu_law_decode(wave_generated_tokens, quantization_channels=256)

assert torch.isfinite(wave_generated).all()
assert torch.max(torch.abs(wave_generated)) <= 1.0001
print("Wave generation finite/range checks passed.")

plt.figure(figsize=(8, 3))
plt.plot(wave_losses)
plt.title("TinyWaveAR training loss")
plt.xlabel("Step")
plt.ylabel("Cross-entropy")
plt.grid(alpha=0.25)
plt.show()


## 6. Conditioning and Control

In practice, music models expose control signals such as:

- Tempo / BPM bins
- Style tags (e.g., piano, ambient, EDM)
- Chord progression / key
- Arrangement cues (intro, drop, bridge)

A minimal way to teach conditioning is to prepend control tokens to sequence prefixes.


In [ ]:
def generate_conditioned_symbolic_dataset(
    num_samples: int,
    seq_len: int,
    base_vocab_size: int,
) -> torch.Tensor:
    """Add synthetic tempo/style control tokens as prefixes."""
    rng = np.random.default_rng(999)

    tempo_slow = base_vocab_size
    tempo_fast = base_vocab_size + 1
    style_smooth = base_vocab_size + 2
    style_staccato = base_vocab_size + 3

    core_len = seq_len - 2
    base = generate_symbolic_dataset(num_samples, core_len, base_vocab_size)

    conditioned = torch.zeros((num_samples, seq_len), dtype=torch.long)
    for i in range(num_samples):
        seq = base[i].clone()

        slow = bool(rng.integers(0, 2) == 0)
        staccato = bool(rng.integers(0, 2) == 1)

        if slow:
            seq[1::2] = seq[:-1:2]
            t_tok = tempo_slow
        else:
            seq = (seq + 1) % base_vocab_size
            t_tok = tempo_fast

        if staccato:
            seq[::4] = (seq[::4] + 5) % base_vocab_size
            s_tok = style_staccato
        else:
            s_tok = style_smooth

        conditioned[i, 0] = t_tok
        conditioned[i, 1] = s_tok
        conditioned[i, 2:] = seq

    return conditioned


conditioned_vocab = vocab_size + 4
conditioned_data = generate_conditioned_symbolic_dataset(
    num_samples=320,
    seq_len=64,
    base_vocab_size=vocab_size,
)

control_model = TinyMusicTransformer(vocab_size=conditioned_vocab).to(device)
_ = train_symbolic(control_model, conditioned_data, steps=70, lr=3e-3, batch_size=32)

slow_smooth_prompt = torch.tensor([vocab_size, vocab_size + 2, 20, 22], dtype=torch.long)
fast_stacc_prompt = torch.tensor([vocab_size + 1, vocab_size + 3, 20, 22], dtype=torch.long)

slow_out = sample_symbolic(control_model, slow_smooth_prompt, max_new_tokens=28, temperature=0.9, top_k=8)
fast_out = sample_symbolic(control_model, fast_stacc_prompt, max_new_tokens=28, temperature=0.9, top_k=8)

plt.figure(figsize=(10, 3))
plt.plot(slow_out.numpy(), label="slow+smooth", lw=1.5)
plt.plot(fast_out.numpy(), label="fast+staccato", lw=1.2, alpha=0.8)
plt.title("Control-token conditioned generations (toy example)")
plt.xlabel("Step")
plt.ylabel("Token")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 7. Evaluation: Lightweight Metrics + Listening Checklist

No single metric fully captures musical quality. Use a bundle:

- **Structure proxies**: repetition, entropy/diversity.
- **Signal sanity**: clipping and spectral statistics.
- **Human checklist**: coherence, variation, transitions, artifacts.


In [ ]:
def repetition_rate(tokens: np.ndarray, ngram: int = 2) -> float:
    """Fraction of repeated n-grams; higher means more looping/repetition."""
    if len(tokens) < ngram + 1:
        return 0.0
    grams = [tuple(tokens[i : i + ngram]) for i in range(len(tokens) - ngram + 1)]
    unique = len(set(grams))
    return 1.0 - (unique / max(len(grams), 1))


def pitch_class_entropy(tokens: np.ndarray) -> float:
    """Entropy over pitch classes (token mod 12), normalized to [0, 1]."""
    pcs = np.asarray(tokens) % 12
    hist = np.bincount(pcs, minlength=12).astype(np.float64)
    p = hist / max(hist.sum(), 1.0)
    p = p[p > 0]
    entropy = -(p * np.log2(p)).sum()
    return float(entropy / np.log2(12))


def clipping_ratio(wave: np.ndarray, threshold: float = 0.98) -> float:
    """Share of samples close to clipping."""
    return float(np.mean(np.abs(wave) >= threshold))


def spectral_centroid_stats(
    wave: np.ndarray,
    sample_rate: int,
    frame_size: int = 512,
    hop: int = 256,
) -> tuple[float, float]:
    """Mean/std spectral centroid in Hz over short-time frames."""
    if len(wave) < frame_size:
        return 0.0, 0.0

    window = np.hanning(frame_size)
    freqs = np.fft.rfftfreq(frame_size, d=1.0 / sample_rate)
    cents = []

    for i in range(0, len(wave) - frame_size + 1, hop):
        frame = wave[i : i + frame_size] * window
        mag = np.abs(np.fft.rfft(frame))
        denom = mag.sum() + 1e-8
        cents.append(float((mag * freqs).sum() / denom))

    arr = np.array(cents, dtype=np.float64)
    return float(arr.mean()), float(arr.std())


sym_tokens = generated.numpy()
wave_np = wave_generated.numpy()

sym_rep = repetition_rate(sym_tokens, ngram=2)
sym_ent = pitch_class_entropy(sym_tokens)
clip = clipping_ratio(wave_np)
cent_mean, cent_std = spectral_centroid_stats(wave_np, sample_rate=sample_rate)

print(f"Symbolic repetition rate: {sym_rep:.3f}")
print(f"Pitch-class entropy (norm): {sym_ent:.3f}")
print(f"Wave clipping ratio: {clip:.4f}")
print(f"Spectral centroid mean/std (Hz): {cent_mean:.1f} / {cent_std:.1f}")

fig, axes = plt.subplots(2, 2, figsize=(12, 6), constrained_layout=True)
axes[0, 0].plot(sym_tokens, lw=1.2)
axes[0, 0].set_title("Generated symbolic tokens")
axes[0, 0].grid(alpha=0.2)

pcs = sym_tokens % 12
axes[0, 1].hist(pcs, bins=np.arange(13) - 0.5, rwidth=0.85)
axes[0, 1].set_title("Pitch-class histogram")
axes[0, 1].set_xlabel("Pitch class")

axes[1, 0].plot(wave_np[:2400], lw=1.0)
axes[1, 0].set_title("Generated waveform preview")
axes[1, 0].grid(alpha=0.2)

n_fft = min(4096, len(wave_np))
fft = np.abs(np.fft.rfft(wave_np[:n_fft]))
freq = np.fft.rfftfreq(n_fft, d=1.0 / sample_rate)
axes[1, 1].plot(freq, fft)
axes[1, 1].set_xlim(0, 4000)
axes[1, 1].set_title("Magnitude spectrum (preview)")
axes[1, 1].set_xlabel("Hz")

plt.show()


### Listening Checklist (Practical)

When you listen to generations, rate each 1-5:

1. **Musical coherence**: does it sustain a phrase-level idea?
2. **Controlled variation**: avoids both monotony and random jumps.
3. **Timing/groove stability**: rhythm feels intentional.
4. **Timbre quality**: no harsh artifacts/pumping/noise bursts.
5. **Prompt/control alignment**: follows requested style/tempo/feel.


## 8. Failure Modes, Licensing, and Data Considerations

Common failure modes:

- **Mode collapse / looping**: over-repetition of short motifs.
- **Structure drift**: starts coherent, then degrades over long horizons.
- **Artifact bursts**: clipping, unstable transients, noisy tails.
- **Control leakage**: model ignores or only weakly follows control tokens.

Data and legal considerations:

- Track dataset provenance and usage rights.
- Separate training/evaluation sets to avoid memorization leakage.
- Document filtering for copyrighted, low-quality, or mislabeled data.
- In product settings, add output moderation policies and attribution strategy.


## 9. FAANG Interview Questions

### Q1: Why is symbolic generation usually easier to train than raw-audio generation?

**Answer:** Symbolic sequences are shorter, lower-entropy, and more semantically structured. Raw audio needs far more tokens and must model timbre-level detail, increasing compute and optimization difficulty.

### Q2: What does mu-law quantization buy us for audio modeling?

**Answer:** It compresses dynamic range so low-amplitude detail is represented with better resolution, enabling discrete-token autoregressive modeling with manageable vocab size.

### Q3: Why do modern music models often use tokenizers/codecs before sequence modeling?

**Answer:** Direct waveform modeling is too long-horizon. Learned codecs compress audio into shorter discrete streams, making Transformer-scale modeling feasible.

### Q4: How would you evaluate a music model without relying on one single metric?

**Answer:** Combine objective proxies (diversity, repetition, signal sanity), controlled prompt-following tests, and structured human listening studies.

### Q5: What is the core tradeoff between autoregressive and non-autoregressive music decoders?

**Answer:** Autoregressive decoding usually improves local coherence but is slower; non-autoregressive/masked decoding is faster and parallelizable but may require iterative refinement for quality.


## 10. Key Takeaways and Next Study Path

1. Representation choice determines what your model can express and how expensive it is.
2. Symbolic-first learning gives fast feedback on compositional structure.
3. Audio-token models are closer to realistic generation but require more care in training/eval.
4. Conditioning is easiest to prototype via prepended control tokens.
5. Good evaluation is multi-dimensional: structure, signal quality, and human judgment.

Recommended next steps:

1. Replace synthetic symbolic data with a small cleaned MIDI corpus.
2. Swap tiny models for larger architectures and longer contexts.
3. Add classifier-free guidance style conditioning experiments.
4. Build A/B listening harnesses with blinded human scoring.


## References

- MusicVAE: https://arxiv.org/abs/1803.05428
- Music Transformer: https://arxiv.org/abs/1809.04281
- Jukebox: https://arxiv.org/abs/2005.00341
- MusicLM: https://arxiv.org/abs/2301.11325
- MusicGen: https://arxiv.org/abs/2306.05284
- AudioCraft (MusicGen code): https://github.com/facebookresearch/audiocraft
- MAGNeT: https://arxiv.org/abs/2401.04577
- Stable Audio Open 1.0: https://huggingface.co/stabilityai/stable-audio-open-1.0
- MusicRL: https://arxiv.org/abs/2402.04229
- MusicLM RLHF page: https://google-research.github.io/seanet/musiclm/rlhf/
- Magenta RealTime: https://magenta.tensorflow.org/magenta-realtime
- Live music model (2025): https://arxiv.org/abs/2508.04651
